# All-Lineage Combined scVI + scANVI Pipeline — v1.5 PRODUCTION
**Author:** r2end | **Date:** 2026-03-19 | **Version:** v1.5 PRODUCTION

## Changes over v1.4

| Tag | Fix | Severity |
|-----|-----|----------|
| P0-1 | `join="outer"` replaces `join="inner"`; missing-gene counts filled with 0 (CSR); gene universe = union of all lineages (~57k) | P0 |
| P0-2 | `.raw` now set from the post-outer-join, pre-filter gene space = true union; run log label corrected to `post_concat_union_raw` | P0 |
| P1-1 | `cell_type_input_l3` holds original per-lineage label; `cell_type_final` assigned **after scANVI** from `cell_type_scanvi_pred_pan` | P1 |
| P1-2 | `soft_df.reindex(adata.obs_names)` before `.idxmax`/`.max` — index-aligned write-back | P1 |
| P1-3 | Myeloid L2 map: unmapped classes now raise `ValueError` (consistent with T/NK + B cell) | P1 |
| QRM-32 | `resolve_fullgene_counts_matrix()` tie-break rank: `raw_counts`(1) > `counts`(2) per QRM Sec 16.2; `.raw.X` keeps rank 0 as largest-gene source | QRM |
| P2-1 | Small batches (`<3 cells`) dropped after filter step; not just warned | P2 |
| P2-2 | Thread consistency: `OMP/OPENBLAS/MKL=8` and `sc.settings.n_jobs=8` | P2 |

## Cell 1 — Logging Setup + Imports

In [ ]:
import os, sys
# [P2-2] Set thread env BEFORE all other imports (BLAS pool init)
os.environ["OMP_NUM_THREADS"]      = "8"
os.environ["OPENBLAS_NUM_THREADS"] = "8"
os.environ["MKL_NUM_THREADS"]      = "8"

import matplotlib
matplotlib.use("Agg")

import gc, time, json, anndata
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import scipy.sparse as sparse
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
import torch

class _Tee:
    def __init__(self, original, log_fh):
        self._orig = original; self._log = log_fh
    def write(self, data):
        self._orig.write(data); self._orig.flush()
        try: self._log.write(data); self._log.flush()
        except Exception: pass
    def flush(self):
        self._orig.flush()
        try: self._log.flush()
        except Exception: pass
    def fileno(self):  return self._orig.fileno()
    def isatty(self):  return False

_LOG_BASE = Path("/home/h2048/data/py/20260319/allcells_combined_scanvi/logs")
_LOG_BASE.mkdir(parents=True, exist_ok=True)
_TS      = datetime.now().strftime("%Y%m%d_%H%M%S")
_NB_STEM = "allcells_combined_scanvi_20260319_v1_5_PRODUCTION"
LOG_FILE = _LOG_BASE / f"{_NB_STEM}_{_TS}.log"
_log_fh  = open(LOG_FILE, "w", encoding="utf-8", buffering=1)
sys.stdout = _Tee(sys.__stdout__, _log_fh)
sys.stderr = _Tee(sys.__stderr__, _log_fh)

anndata.settings.allow_write_nullable_strings = True  # [QRM-33 / v3.5 §17.1]
GPU_AVAILABLE = torch.cuda.is_available()
_accelerator  = "gpu" if GPU_AVAILABLE else "cpu"
_devices      = 1     if GPU_AVAILABLE else "auto"
scvi.settings.seed = 42
if GPU_AVAILABLE: torch.cuda.manual_seed_all(42)
np.random.seed(42)
sc.settings.n_jobs    = 8   # [P2-2] match OMP_NUM_THREADS
sc.settings.verbosity = 2
PIPELINE_START = time.time()

print("=" * 70)
print(f"Pipeline   : {_NB_STEM}")
print(f"Log file   : {LOG_FILE}")
print(f"Start      : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"scvi-tools : {scvi.__version__}")
print(f"scanpy     : {sc.__version__}")
print(f"Accelerator: {_accelerator}  devices={_devices}")
print("=" * 70)
print(f"  tail -f {LOG_FILE}")

/home/h2048/miniconda3/envs/scvi_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Seed set to 42


Pipeline   : allcells_combined_scanvi_20260319_v1_5_PRODUCTION
Log file   : /home/h2048/data/py/20260319/allcells_combined_scanvi/logs/allcells_combined_scanvi_20260319_v1_5_PRODUCTION_20260322_161621.log
Start      : 2026-03-22 16:16:22
scvi-tools : 1.3.3
scanpy     : 1.11.5
Accelerator: gpu  devices=1
  tail -f /home/h2048/data/py/20260319/allcells_combined_scanvi/logs/allcells_combined_scanvi_20260319_v1_5_PRODUCTION_20260322_161621.log
Output     : allcells_combined_20260319_v1_5.h5ad
Batch key  : sample  (sample-level unified)
concat     : outer join + zero-fill  (P0-1 fix)
ALLOW_UNKNOWN=False  DROP_UNKNOWN=True
MIN_CELLS_PER_BATCH=3
  epithelial  : epithelial_scanvi_v2_7_HOTFIX_SELF_final.h5ad
  tcell       : adata_tnk_scanvi_ref_retrain_v1_2.h5ad
  myeloid     : adata_myeloid_refined_FINAL.h5ad
  bcell       : bcell_reference_20260203.h5ad
  stromal     : stromal_reintegrated_scvi_scanvi_v1_3.h5ad


/tmp/ipykernel_11985/337905217.py:60: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print(f"scanpy     : {sc.__version__}")


[OK] Label maps and builder functions defined
[OK] Structural helpers defined

[STEP 5] Loading, building labels, normalizing

[epithelial] epithelial_scanvi_v2_7_HOTFIX_SELF_final.h5ad
  shape : 278,380 x 4,024 | .raw : 53923 genes
  [label build] epithelial
    override: Hillock-like n=1,547
    override: Ionocyte n=155
    override: Neuroendocrine n=27
    -> L3: 24 classes  L2: 6 classes  overrides: 1,729
  [normalize] epithelial
    counts: CSR float32 (278380, 4024)
    batch: 'sample' present (150 levels)
    obs: dropped scVI internal cols: ['_scvi_batch', '_scvi_labels']
    var: dropped HVG aux cols: ['highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches']
    obsm: cleared 10 embeddings
  [WARNING] epithelial: adata.raw.X non-integer (max_frac=0.5000)
  [OK] epithelial: adata.raw.X | genes=53,923
  -> 278,380 x 53,923 | 24 L3 classes | src='adata.raw.X'

[tcell] adata_tnk_scanvi_ref_retrain_v1_2.h5ad
  shape : 38,904 x 4

/home/h2048/miniconda3/envs/scvi_env/lib/python3.10/site-packages/legacy_api_wrap/__init__.py:88: UserWarning: `flavor='seurat_v3'` expects raw count data, but non-integers were found.
  return fn(*args_all, **kw)


...

Epoch 400/400: 100%|##########| 400/400 [4:21:36<00:00, 39.24s/it, v_num=1, train_loss_step=486, train_loss_epoch=461]
INFO     File /home/h2048/data/py/20260319/allcells_combined_scanvi/models/scvi_model/model.pt already downloaded  
INFO     Found 100.0% reference vars in query data.                                                                
[OK] scVI scArches dry-run (2k)
[OK] scVI saved: /home/h2048/data/py/20260319/allcells_combined_scanvi/models/scvi_model

[STEP 16] Neighbors + UMAP (scVI)
computing neighbors
    finished (0:02:20)
computing UMAP
    finished (0:11:03)
[OK] X_umap_scvi saved

[STEP 17] scANVI training (L3 fully supervised)


/home/h2048/miniconda3/envs/scvi_env/lib/python3.10/site-packages/scvi/data/fields/_base_field.py:63: UserWarning: adata.layers[counts] does not contain unnormalized count data. Are you sure this is what you want?
  self.validate_field(adata)
/home/h2048/miniconda3/envs/scvi_env/lib/python3.10/site-packages/scvi/data/fields/_dataframe_field.py:187: UserWarning: Category 8 in adata.obs['_scvi_labels'] has fewer than 3 cells. Models may not train properly.
  categorical_mapping = _make_column_categorical(
/home/h2048/miniconda3/envs/scvi_env/lib/python3.10/site-packages/scvi/data/fields/_scanvi.py:56: UserWarning: Category 8 in adata.obs['_scvi_labels'] has fewer than 3 cells. Models may not train properly.
  mapping = _make_column_categorical(


INFO     Training for 200 epochs.                                                                                  


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/h2048/miniconda3/envs/scvi_env/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=111` in the `DataLoader` to improve performance.
/home/h2048/miniconda3/envs/scvi_env/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=111` in the `DataLoader` to improve performance.


Epoch 87/200:  44%|####3     | 87/200 [2:08:40<2:47:07, 88.74s/it, v_num=1, train_loss_step=506, train_loss_epoch=442]
Monitored metric elbo_validation did not improve in the last 45 records. Best score: 501.746. Signaling Trainer to stop.
[OK] scANVI: 108 L3 classes | mean confidence=0.8776
[P1-1] cell_type_final <- cell_type_scanvi_pred_pan (pan-scANVI refined)
       cell_type_input_l3 retains original per-lineage labels
INFO     File /home/h2048/data/py/20260319/allcells_combined_scanvi/models/scanvi_model/model.pt already downloaded
INFO     Found 100.0% reference vars in query data.                                                                
[OK] scANVI scArches dry-run (2k)
[OK] scANVI saved: /home/h2048/data/py/20260319/allcells_combined_scanvi/models/scanvi_model

[STEP 18] Neighbors + UMAP (scANVI) + agreement check
======================================================================computing neighbors

    finished (0:01:28)
computing UMAP
    finished (0:10:53)
Overal

## Cell 2 — Configuration

In [2]:
DATE_TAG   = "20260319"
VERSION    = "1_5"
OUTPUT_DIR = Path(f"/home/h2048/data/py/{DATE_TAG}/allcells_combined_scanvi")
MODEL_DIR  = OUTPUT_DIR / "models"
FIG_DIR    = OUTPUT_DIR / "figures"
for d in [OUTPUT_DIR, MODEL_DIR, FIG_DIR, _LOG_BASE]:
    d.mkdir(parents=True, exist_ok=True)

# [P0-1] join="outer" => need all lineages on same gene universe.
# cell_type_L3 / cell_type_L2 are built by per-lineage builder functions below.
LINEAGE_INPUTS = {
    "epithelial": {
        "h5ad"           : Path("/home/h2048/data/py/0317/epithelial_v2_7_HOTFIX/SELF/"
                                "epithelial_scanvi_v2_7_HOTFIX_SELF_final.h5ad"),
        "label"          : "cell_type_L3",
        "name"           : "Epithelial",
        "batch_key_local": "sample",
    },
    "tcell": {
        "h5ad"           : Path("/home/h2048/data/py/0318/tnk_subcluster_retrain/"
                                "adata_tnk_scanvi_ref_retrain_v1_2.h5ad"),
        "label"          : "cell_type_L3",
        "name"           : "TNK",
        "batch_key_local": "sample",
    },
    "myeloid": {
        "h5ad"           : Path("/home/h2048/data/py/0209/myeloid_validation_optimized/"
                                "adata_myeloid_refined_FINAL.h5ad"),
        "label"          : "cell_type_L3",
        "name"           : "Myeloid",
        "batch_key_local": "sample",
    },
    "bcell": {
        "h5ad"           : Path("/home/h2048/data/py/0203/bcell_scarches_v4_1/results/"
                                "scarches_package/bcell_reference_20260203.h5ad"),
        "label"          : "cell_type_L3",
        "name"           : "Bcell",
        "batch_key_local": "sample",
    },
    "stromal": {
        "h5ad"           : Path("/home/h2048/data/py/0308/stromal_reintegration_v1_3/"
                                "stromal_reintegrated_scvi_scanvi_v1_3.h5ad"),
        "label"          : "cell_type_L3",
        "name"           : "Stromal",
        "batch_key_local": "sample",
    },
}

BATCH_KEY                  = "sample"
LABELS_KEY                 = "scanvi_label"
UNLABELED_CATEGORY         = "Unknown"
N_HVG                      = 4000
N_LATENT                   = 100
SCVI_EPOCHS                = 400
SCANVI_EPOCHS              = 200
ALLOW_UNKNOWN              = False
DROP_UNKNOWN_IF_DISALLOWED = True
MIN_CELLS_PER_BATCH        = 3    # [P2-1] batches below this are dropped

OUTPUT_H5AD   = OUTPUT_DIR / f"allcells_combined_{DATE_TAG}_v{VERSION}.h5ad"
PIPELINE_NAME = f"allcells_combined_v{VERSION}"

for key, cfg in LINEAGE_INPUTS.items():
    p = cfg["h5ad"]
    if not p.exists():
        hits = sorted(Path("/home/h2048/data").rglob(p.name))[:5]
        raise FileNotFoundError(f"[X] {key}: {p}\n    Matches: {hits}")

print(f"Output     : {OUTPUT_H5AD.name}")
print(f"Batch key  : {BATCH_KEY}  (sample-level unified)")
print(f"concat     : outer join + zero-fill  (P0-1 fix)")
print(f"ALLOW_UNKNOWN={ALLOW_UNKNOWN}  DROP_UNKNOWN={DROP_UNKNOWN_IF_DISALLOWED}")
print(f"MIN_CELLS_PER_BATCH={MIN_CELLS_PER_BATCH}")
for key, cfg in LINEAGE_INPUTS.items():
    print(f"  {key:12s}: {cfg['h5ad'].name}")

## Cell 3 — Per-Lineage Label Maps + Builder Functions

In [6]:
# ════════════════════════════════════════════════════════════════
# T/NK  maps
# ════════════════════════════════════════════════════════════════
_TNK_L3_RENAME = {"CD4 Naive/TCM": "CD4 Naive"}

_TNK_L2_MAP = {
    "CD8 Trm"     : "CD8 T cells", "CD8 Tem"     : "CD8 T cells",
    "CD8 Temra"   : "CD8 T cells", "CD8 Teff"    : "CD8 T cells",
    "CD8 Naive"   : "CD8 T cells",
    "NK"          : "NK cells",    "NK Exhausted" : "NK cells",
    "CD4 Trm"     : "CD4 T cells", "CD4 Tcm"     : "CD4 T cells",
    "CD4 Tfh"     : "CD4 T cells", "CD4 Tfr"     : "CD4 T cells",
    "CD4 Treg"    : "CD4 T cells", "CD4 Th17"    : "CD4 T cells",
    "CD4 Th1"     : "CD4 T cells", "CD4 Naive"   : "CD4 T cells",
    "gdT"         : "gdT cells",   "MAIT"         : "MAIT cells",
    "ILC3"        : "ILC",
}

# ════════════════════════════════════════════════════════════════
# Myeloid  maps
# ════════════════════════════════════════════════════════════════
_MYELOID_FINEST_L2 = {
    "Alveolar macrophages"          : "Alveolar_Macrophage",
    "Alveolar Mph CCL3+"            : "Alveolar_Macrophage",
    "Alveolar Mph proliferating"    : "Alveolar_Macrophage",
    "Alveolar Mph MT-positive"      : "Alveolar_Macrophage",
    "Monocyte-derived Mph"          : "Interstitial_Macrophage",
    "Interstitial Mph perivascular" : "Interstitial_Macrophage",
    "Classical monocytes"           : "Monocyte",
    "Non-classical monocytes"       : "Monocyte",
    "DC2"                           : "DC",
    "Plasmacytoid DCs"              : "DC",
    "Mast cells"                    : "Mast_cell",
}
_MYELOID_REFINED_L2 = {
    "Resident Alveolar macrophages"                : "Alveolar_Macrophage",
    "Resting Alveolar macrophages"                 : "Alveolar_Macrophage",
    "Inflammatory Interstitial macrophages"        : "Interstitial_Macrophage",
    "M2-like Interstitial macrophages"             : "Interstitial_Macrophage",
    "Atypically activated Interstitial macrophages": "Interstitial_Macrophage",
    "CD163L1+ Interstitial macrophages"            : "Interstitial_Macrophage",
    "Low-quality Interstitial macrophages"         : "Interstitial_Macrophage",
    "Immunoregulatory Interstitial macrophages"    : "Interstitial_Macrophage",
    "Neutrophils"                                  : "Neutrophil",
    "Typical Classical monocytes"                  : "Monocyte",
    "Inflammatory Classical monocytes"             : "Monocyte",
    "Conventional cDC2"                            : "DC",
    "Langerhans-like cDC2"                         : "DC",
    "pDC"                                          : "DC",
    "Mast cells"                                   : "Mast_cell",
}

# Non-myeloid contaminants appearing in ann_finest_level
# These are doublets / cross-lineage contamination -> map to Unknown -> will be dropped
_MYELOID_CONTAMINANT_L2 = {
    # Epithelial
    "AT2"                      : "Unknown",
    "Basal resting"            : "Unknown",
    "Club (nasal)"             : "Unknown",
    "Club (non-nasal)"         : "Unknown",
    "Goblet (nasal)"           : "Unknown",
    # Stromal / Vascular
    "Adventitial fibroblasts"  : "Unknown",
    "Alveolar fibroblasts"     : "Unknown",
    "Smooth muscle"            : "Unknown",
    "Lymphatic EC mature"      : "Unknown",
    # Immune (non-myeloid)
    "B cells"                  : "Unknown",
    "Plasma cells"             : "Unknown",
    "CD4 T cells"              : "Unknown",
    "CD8 T cells"              : "Unknown",
    "Hematopoietic stem cells" : "Unknown",
    # DC subtypes not in primary map
    "DC1"                      : "DC",
    "Migratory DCs"            : "DC",
}

_MYELOID_ALL_L2 = {**_MYELOID_FINEST_L2, **_MYELOID_REFINED_L2, **_MYELOID_CONTAMINANT_L2}

# ════════════════════════════════════════════════════════════════
# B cell  maps
# ════════════════════════════════════════════════════════════════
_BCELL_L3_MERGE = {"IGHEplus_Atypical_Memory_B": "Atypical_Memory_B"}
_BCELL_L2_MAP   = {
    "GC_B_Dark_Zone_Centroblast_Cycling": "GC_B",
    "GC_B_Light_Zone_Centrocyte"        : "GC_B",
    "GC_B_Transitional"                 : "GC_B",
    "Plasma_IgA"                        : "Plasma",
    "Plasma_IgG"                        : "Plasma",
    "Atypical_Memory_B"                 : "Atypical_Memory_B",
    "Memory_B"                          : "Memory_B",
    "Naive_B"                           : "Naive_B",
}


# ════════════════════════════════════════════════════════════════
# Builder functions
# ════════════════════════════════════════════════════════════════

def build_epithelial_labels(adata_lin):
    """
    L3 <- scanvi_fine_pred
    L2 <- scanvi_major_pred
    Barcode-level overrides for Rare_Specialized:
      ann_finest_level == 'Hillock-like'              -> L3='Hillock-like', L2='Rare_Specialized'
      ann_coarse_for_GWAS_and_modeling == 'Ionocyte'      -> L3, L2
      ann_coarse_for_GWAS_and_modeling == 'Neuroendocrine'-> L3, L2
    """
    assert "scanvi_fine_pred"  in adata_lin.obs.columns, "[X] epithelial: missing scanvi_fine_pred"
    assert "scanvi_major_pred" in adata_lin.obs.columns, "[X] epithelial: missing scanvi_major_pred"

    l3 = adata_lin.obs["scanvi_fine_pred"].astype(str).copy()
    l2 = adata_lin.obs["scanvi_major_pred"].astype(str).copy()
    n_overrides = 0

    if "ann_finest_level" in adata_lin.obs.columns:
        mask = adata_lin.obs["ann_finest_level"].astype(str) == "Hillock-like"
        l3[mask] = "Hillock-like"; l2[mask] = "Rare_Specialized"
        n_overrides += int(mask.sum())
        print(f"    override: Hillock-like n={mask.sum():,}")

    if "ann_coarse_for_GWAS_and_modeling" in adata_lin.obs.columns:
        coarse = adata_lin.obs["ann_coarse_for_GWAS_and_modeling"].astype(str)
        for rare in ["Ionocyte", "Neuroendocrine"]:
            mask = coarse == rare
            l3[mask] = rare; l2[mask] = "Rare_Specialized"
            n_overrides += int(mask.sum())
            print(f"    override: {rare} n={mask.sum():,}")

    adata_lin.obs["cell_type_L3"] = l3
    adata_lin.obs["cell_type_L2"] = l2
    print(f"    -> L3: {l3.nunique()} classes  L2: {l2.nunique()} classes  overrides: {n_overrides:,}")


def build_tcell_labels(adata_lin):
    """L3 <- scanvi_label_refined (CD4 Naive/TCM renamed); L2 by map."""
    assert "scanvi_label_refined" in adata_lin.obs.columns,         "[X] tcell: missing scanvi_label_refined"
    l3 = adata_lin.obs["scanvi_label_refined"].astype(str).copy().replace(_TNK_L3_RENAME)
    unmapped = set(l3.unique()) - set(_TNK_L2_MAP.keys())
    if unmapped:
        raise ValueError(f"[X] tcell: unmapped L3 -> L2: {sorted(unmapped)}")
    adata_lin.obs["cell_type_L3"] = l3
    adata_lin.obs["cell_type_L2"] = l3.map(_TNK_L2_MAP)
    print(f"    -> L3: {l3.nunique()} classes  L2: {adata_lin.obs['cell_type_L2'].nunique()} classes")


def build_myeloid_labels(adata_lin):
    """
    L3 primary  <- ann_finest_level (skip Unknown/nan)
    L3 fallback <- cell_type_L3_refined
    L2          <- combined map. [P1-3] unmapped -> ValueError.
    """
    assert "ann_finest_level"     in adata_lin.obs.columns, "[X] myeloid: missing ann_finest_level"
    assert "cell_type_L3_refined" in adata_lin.obs.columns, "[X] myeloid: missing cell_type_L3_refined"

    finest  = adata_lin.obs["ann_finest_level"].astype(str)
    refined = adata_lin.obs["cell_type_L3_refined"].astype(str)
    _bad    = {"Unknown", "nan", "NaN", "None", ""}
    use_finest = ~finest.isin(_bad)
    l3 = finest.where(use_finest, refined)
    l3 = l3.replace({"nan": "Unknown", "NaN": "Unknown", "None": "Unknown"})

    n_from_finest  = int(use_finest.sum())
    n_from_refined = int((~use_finest).sum())
    n_unknown_l3   = int((l3 == "Unknown").sum())
    print(f"    L3 source: finest={n_from_finest:,}  refined={n_from_refined:,}  Unknown={n_unknown_l3:,}")

    # [P1-3] Hard-fail on unmapped (consistent with TNK + Bcell)
    non_unknown = set(l3[l3 != "Unknown"].unique())
    unmapped    = non_unknown - set(_MYELOID_ALL_L2.keys())
    if unmapped:
        raise ValueError(
            f"[X] myeloid: unmapped L3 -> L2 (add to _MYELOID_FINEST_L2 or _MYELOID_REFINED_L2):\n"
            f"    {sorted(unmapped)}"
        )

    l2 = l3.map(_MYELOID_ALL_L2).fillna("Unknown")   # only Unknown cells map to Unknown
    adata_lin.obs["cell_type_L3"] = l3
    adata_lin.obs["cell_type_L2"] = l2
    print(f"    -> L3: {l3.nunique()} classes  L2: {l2.nunique()} classes")


def build_bcell_labels(adata_lin):
    """L3 <- cell_type_scanvi_pred (IGHEplus merged); L2 by map."""
    assert "cell_type_scanvi_pred" in adata_lin.obs.columns,         "[X] bcell: missing cell_type_scanvi_pred"
    l3 = adata_lin.obs["cell_type_scanvi_pred"].astype(str).copy().replace(_BCELL_L3_MERGE)
    unmapped = set(l3.unique()) - set(_BCELL_L2_MAP.keys())
    if unmapped:
        raise ValueError(f"[X] bcell: unmapped L3 -> L2: {sorted(unmapped)}")
    adata_lin.obs["cell_type_L3"] = l3
    adata_lin.obs["cell_type_L2"] = l3.map(_BCELL_L2_MAP)
    print(f"    -> L3: {l3.nunique()} classes  L2: {adata_lin.obs['cell_type_L2'].nunique()} classes")
    print(f"       L3: {l3.value_counts().to_dict()}")


def build_stromal_labels(adata_lin):
    """L3 <- cell_type_scanvi_pred; L2 <- cell_type_L2 (already present)."""
    assert "cell_type_scanvi_pred" in adata_lin.obs.columns, "[X] stromal: missing cell_type_scanvi_pred"
    assert "cell_type_L2"          in adata_lin.obs.columns, "[X] stromal: missing cell_type_L2"
    adata_lin.obs["cell_type_L3"] = adata_lin.obs["cell_type_scanvi_pred"].astype(str)
    adata_lin.obs["cell_type_L2"] = adata_lin.obs["cell_type_L2"].astype(str)
    print(f"    -> L3: {adata_lin.obs['cell_type_L3'].nunique()} classes  "
          f"L2: {adata_lin.obs['cell_type_L2'].nunique()} classes")


_LABEL_BUILDERS = {
    "epithelial": build_epithelial_labels,
    "tcell"     : build_tcell_labels,
    "myeloid"   : build_myeloid_labels,
    "bcell"     : build_bcell_labels,
    "stromal"   : build_stromal_labels,
}

print("[OK] Label maps and builder functions defined")

## Cell 4 — Structural Normalization + Counts Resolution Helpers

In [4]:
_SCVI_INTERNAL_OBS_COLS = {
    "_scvi_batch", "_scvi_labels",
    "_scvi_extra_categorical_covs", "_scvi_extra_continuous_covs",
}
_HVG_AUX_VAR_COLS = {
    "highly_variable", "highly_variable_rank", "means", "variances",
    "variances_norm", "dispersions", "dispersions_norm", "highly_variable_nbatches",
}


def normalize_lineage_structure(adata_lin, key, local_bk, global_bk):
    """
    Standardize lineage AnnData in-place (v1.3.1 design, QRM Sec 16.x):
    1. counts layer  : ensure CSR float32; create from raw_counts/.raw if absent
    2. batch column  : unify local -> global
    3. obs cleanup   : scVI internal cols removed; string-backed category -> object
    4. var cleanup   : stale HVG aux cols removed (QRM Sec 12.5)
    5. obsm cleanup  : all lineage embeddings cleared
    """
    print(f"  [normalize] {key}")

    # 1. counts layer
    if "counts" not in adata_lin.layers:
        if "raw_counts" in adata_lin.layers:
            adata_lin.layers["counts"] = adata_lin.layers["raw_counts"]
            print(f"    counts: from layers['raw_counts']")
        elif adata_lin.raw is not None:
            raw_idx = pd.Index(adata_lin.raw.var_names)
            cur_idx = pd.Index(adata_lin.var_names)
            shared  = raw_idx.intersection(cur_idx)
            if len(shared) == len(cur_idx):
                pos = raw_idx.get_indexer(cur_idx)
                adata_lin.layers["counts"] = sparse.csr_matrix(
                    adata_lin.raw.X[:, pos], dtype=np.float32)
                print(f"    counts: aligned from .raw.X ({len(shared)} genes)")
            else:
                raise ValueError(
                    f"[X] {key}: counts absent; .raw covers {len(shared)}/{len(cur_idx)} genes")
        else:
            raise ValueError(f"[X] {key}: no counts source")

    if not sparse.isspmatrix_csr(adata_lin.layers["counts"]):
        adata_lin.layers["counts"] = sparse.csr_matrix(
            adata_lin.layers["counts"], dtype=np.float32)
    elif adata_lin.layers["counts"].dtype != np.float32:
        adata_lin.layers["counts"] = adata_lin.layers["counts"].astype(np.float32)

    # [QRM 16.1] Global finite/negative check on .data array
    _d = adata_lin.layers["counts"].data
    if len(_d) > 0:
        if not np.isfinite(_d).all():
            raise ValueError(f"[X] {key}: non-finite in counts")
        if np.any(_d < 0):
            raise ValueError(f"[X] {key}: negative in counts")
    print(f"    counts: CSR float32 {adata_lin.layers['counts'].shape}")

    # 2. batch column
    if global_bk not in adata_lin.obs.columns:
        if local_bk in adata_lin.obs.columns:
            adata_lin.obs[global_bk] = adata_lin.obs[local_bk].astype(object)
            print(f"    batch: '{local_bk}' -> '{global_bk}' ({adata_lin.obs[global_bk].nunique()} levels)")
        else:
            cands = [c for c in adata_lin.obs.columns
                     if any(k in c.lower() for k in ("sample","batch","dataset"))]
            raise ValueError(f"[X] {key}: batch col not found. Candidates: {cands}")
    else:
        adata_lin.obs[global_bk] = adata_lin.obs[global_bk].astype(object)
        print(f"    batch: '{global_bk}' present ({adata_lin.obs[global_bk].nunique()} levels)")

    # 3. obs cleanup
    drop_obs = [c for c in adata_lin.obs.columns if c in _SCVI_INTERNAL_OBS_COLS]
    if drop_obs:
        adata_lin.obs.drop(columns=drop_obs, inplace=True)
        print(f"    obs: dropped scVI internal cols: {drop_obs}")
    for col in adata_lin.obs.select_dtypes(include=["category"]).columns:
        cats = adata_lin.obs[col].cat.categories
        if hasattr(cats.dtype, "name") and cats.dtype.name in ("string","StringDtype"):
            adata_lin.obs[col] = adata_lin.obs[col].cat.rename_categories(cats.astype(object))

    # 4. var cleanup [QRM 12.5]
    drop_var = [c for c in adata_lin.var.columns if c in _HVG_AUX_VAR_COLS]
    if drop_var:
        adata_lin.var.drop(columns=drop_var, inplace=True)
        print(f"    var: dropped HVG aux cols: {drop_var}")

    # 5. obsm cleanup
    n_obsm = len(adata_lin.obsm)
    if n_obsm:
        adata_lin.obsm.clear()
        print(f"    obsm: cleared {n_obsm} embeddings")


def resolve_fullgene_counts_matrix(adata_lin, key):
    """
    Select counts source with most genes; return CSR float32.

    Tie-break rank (QRM Sec 16.2):
      .raw.X     -> rank 0  (may have most genes; used only if n_vars > others)
      raw_counts -> rank 1  (QRM 16.2: preferred over counts when tied)
      counts     -> rank 2
    Checks: non-finite, negative. Soft integer warning (no hard fail).
    """
    candidates = []
    # [QRM 16.2] raw_counts preferred over counts on tie
    if adata_lin.raw is not None:
        candidates.append(dict(source="adata.raw.X", X=adata_lin.raw.X,
                               var=adata_lin.raw.var.copy(),
                               n_vars=adata_lin.raw.n_vars, rank=0))
    if "raw_counts" in adata_lin.layers:
        candidates.append(dict(source="layers['raw_counts']", X=adata_lin.layers["raw_counts"],
                               var=adata_lin.var.copy(), n_vars=adata_lin.n_vars, rank=1))
    if "counts" in adata_lin.layers:
        candidates.append(dict(source="layers['counts']", X=adata_lin.layers["counts"],
                               var=adata_lin.var.copy(), n_vars=adata_lin.n_vars, rank=2))

    if not candidates:
        raise ValueError(f"[X] {key}: no counts source")
    candidates.sort(key=lambda c: (-c["n_vars"], c["rank"]))

    skipped = []
    for cand in candidates:
        X, src = cand["X"], cand["source"]
        _d = X.data if sparse.issparse(X) else np.asarray(X).ravel()
        if len(_d) > 0 and not np.isfinite(_d).all():
            skipped.append(f"    [SKIP] {src}: non-finite"); continue
        if len(_d) > 0 and np.any(_d < 0):
            skipped.append(f"    [SKIP] {src}: negative"); continue
        # Soft integer warning
        if len(_d) > 0:
            _s = _d[:min(50000, len(_d))]
            frac = np.abs(_s - np.round(_s))
            if np.nanmax(frac) > 1e-3:
                print(f"  [WARNING] {key}: {src} non-integer (max_frac={np.nanmax(frac):.4f})")
        if skipped:
            print("  [INFO] skipped:"); [print(s) for s in skipped]
        if src != "adata.raw.X" and adata_lin.raw is not None:
            print(f"  [WARNING] {key}: .raw not used ({adata_lin.raw.n_vars:,} genes); using {src}")
        # [v1.3.1-2] CSR float32
        if sparse.issparse(X):
            if not sparse.isspmatrix_csr(X): X = X.tocsr()
            if X.dtype != np.float32:        X = X.astype(np.float32)
        else:
            X = sparse.csr_matrix(np.asarray(X, dtype=np.float32))
        print(f"  [OK] {key}: {src} | genes={cand['n_vars']:,}")
        return X, src, cand["var"]

    raise ValueError(f"[X] {key}: all candidates failed. " + " | ".join(skipped))


def _fmt_float(val, fallback="not_set"):
    return f"{val:.4f}" if isinstance(val, (float, np.floating)) else str(fallback)


print("[OK] Structural helpers defined")

## Cell 5 — Load Lineages + Build L3/L2 Labels + Normalize

In [7]:
print("\n" + "="*70)
print("[STEP 5] Loading, building labels, normalizing")
print("="*70)

lineage_adatas = []
label_sets     = {}
gene_sets      = {}

for key, cfg in LINEAGE_INPUTS.items():
    print(f"\n[{key}] {cfg['h5ad'].name}")
    adata_lin = sc.read_h5ad(cfg["h5ad"])
    print(f"  shape : {adata_lin.n_obs:,} x {adata_lin.n_vars:,} | "
          f".raw : {adata_lin.raw.n_vars if adata_lin.raw is not None else 'None'} genes")

    print(f"  [label build] {key}")
    _LABEL_BUILDERS[key](adata_lin)

    normalize_lineage_structure(
        adata_lin, key,
        local_bk  = cfg.get("batch_key_local", BATCH_KEY),
        global_bk = BATCH_KEY,
    )

    counts_X, count_source, var_df = resolve_fullgene_counts_matrix(adata_lin, key)

    adata_full = sc.AnnData(
        X   = counts_X,
        obs = adata_lin.obs.copy(),
        var = var_df,
    )
    adata_full.layers["counts"]             = adata_full.X    # shared ref, no copy
    adata_full.obs["lineage_source"]        = cfg["name"]
    adata_full.obs["cell_type_original"]    = (
        adata_lin.obs["cell_type_L3"].astype(object).fillna(UNLABELED_CATEGORY)
    )
    adata_full.obs["cell_type_L2_original"] = (
        adata_lin.obs["cell_type_L2"].astype(object).fillna("Unknown")
    )

    label_sets[key] = set(adata_full.obs["cell_type_original"].unique())
    gene_sets[key]  = set(adata_full.var_names.tolist())

    print(f"  -> {adata_full.n_obs:,} x {adata_full.n_vars:,} | "
          f"{len(label_sets[key])} L3 classes | src='{count_source}'")

    lineage_adatas.append(adata_full)
    del adata_lin; gc.collect()

print("\n[OK] All lineages loaded")

## Cell 6 — Pre-concat Audits (label overlap + gene universe)

In [8]:
print("\n" + "="*70)
print("[STEP 6] Pre-concat audits")
print("="*70)

print("\n[6a] Cross-lineage L3 label overlap")
all_keys = list(label_sets.keys())
for i in range(len(all_keys)):
    for j in range(i + 1, len(all_keys)):
        ka, kb = all_keys[i], all_keys[j]
        overlap = sorted((label_sets[ka] & label_sets[kb]) - {UNLABELED_CATEGORY})
        if overlap:
            print(f"  [INFO] {ka} x {kb}: {len(overlap)} shared: {overlap}")
            print(f"         -> same label name = ONE class in pan-lineage scANVI")
        else:
            print(f"  [OK]   {ka} x {kb}: no overlap")

print("\n[6b] Gene universe consistency")
for k in gene_sets:
    print(f"  {k:12s}: {len(gene_sets[k]):,} genes")
intersection = set.intersection(*gene_sets.values())
union        = set.union(*gene_sets.values())
coverage     = len(intersection) / max(len(union), 1)
print(f"\n  Inner-join would give : {len(intersection):,} genes")
print(f"  Outer-join (actual)   : {len(union):,} genes  [P0-1 fix: using outer]")
print(f"  Inner/outer coverage  : {coverage:.3f}")
print(f"  -> Outer join retains all lineage-specific genes; missing entries filled 0")

## Cell 7 — Concatenate (outer join + zero-fill)

In [9]:
print("\n" + "="*70)
print("[STEP 7] Concatenating with outer join (P0-1 fix)")
print("="*70)

# [P0-1] join="outer" preserves the full gene universe of all lineages.
# Missing genes in each lineage's counts become explicit zeros in sparse matrix.
adata = sc.concat(
    lineage_adatas,
    join         = "outer",   # [P0-1] was "inner"; now union gene space
    merge        = "unique",
    uns_merge    = "unique",
    label        = "lineage_concat_key",
    keys         = list(LINEAGE_INPUTS.keys()),
    index_unique = "-",
    fill_value   = 0,         # NaN for missing genes -> 0 (absent = unexpressed)
)
del lineage_adatas; gc.collect()

print(f"Combined: {adata.n_obs:,} x {adata.n_vars:,}")
print(f"Gene universe: {adata.n_vars:,} (outer join = union of all lineages)")
print(f"\nLineage distribution:")
print(adata.obs["lineage_source"].value_counts().to_string())

# [P0-1] After outer join, counts may have float NaN from fill_value in some
# backends. Explicitly convert to CSR float32 and ensure no non-finite values.
print("\n[P0-1] Ensuring counts layer is clean CSR float32 after outer join...")
if "counts" in adata.layers:
    X_counts = adata.layers["counts"]
    if not sparse.issparse(X_counts):
        # Dense after outer join -- convert to sparse, replacing NaN -> 0
        X_arr = np.asarray(X_counts, dtype=np.float32)
        X_arr = np.where(np.isfinite(X_arr), X_arr, 0.0)
        X_arr = np.where(X_arr < 0, 0.0, X_arr)
        adata.layers["counts"] = sparse.csr_matrix(X_arr, dtype=np.float32)
        del X_arr; gc.collect()
    else:
        X_csr = sparse.csr_matrix(X_counts, dtype=np.float32, copy=False)
        # Replace any non-finite values (NaN/inf) with 0
        if not np.isfinite(X_csr.data).all():
            n_bad = int((~np.isfinite(X_csr.data)).sum())
            print(f"  [WARNING] Replacing {n_bad:,} non-finite values with 0")
            X_csr = X_csr.copy()
            X_csr.data[~np.isfinite(X_csr.data)] = 0.0
            X_csr.eliminate_zeros()
        if np.any(X_csr.data < 0):
            n_neg = int(np.sum(X_csr.data < 0))
            raise ValueError(f"[X] {n_neg:,} negative values in counts after outer join")
        adata.layers["counts"] = X_csr
else:
    raise ValueError("[X] counts layer missing after concat")

# Final assertion [QRM 16.1]
_d = adata.layers["counts"].data
assert np.isfinite(_d).all(), "[X] Non-finite in counts after cleanup"
assert np.all(_d >= 0),       "[X] Negative in counts after cleanup"
del _d; gc.collect()
print(f"[OK] counts: CSR float32 {adata.layers['counts'].shape}")
print(f"[OK] counts integrity verified (outer join + zero-fill)")

## Cell 8 — Build `scanvi_label` (L3) + `cell_type_L2` + `cell_type_input_l3`

In [10]:
print("\n" + "="*70)
print("[STEP 8] Unified scanvi_label (L3) + cell_type_L2 + cell_type_input_l3")
print("="*70)

# [P1-1] Original per-lineage label saved under cell_type_input_l3 (not cell_type_final)
# cell_type_final will be assigned AFTER scANVI from cell_type_scanvi_pred_pan.
adata.obs["cell_type_input_l3"] = adata.obs["cell_type_original"].astype(object)

adata.obs[LABELS_KEY] = pd.Categorical(
    adata.obs["cell_type_original"].astype(object).fillna(UNLABELED_CATEGORY)
)
if UNLABELED_CATEGORY not in adata.obs[LABELS_KEY].cat.categories:
    adata.obs[LABELS_KEY] = adata.obs[LABELS_KEY].cat.add_categories([UNLABELED_CATEGORY])

# Unified L2
adata.obs["cell_type_L2"] = pd.Categorical(
    adata.obs["cell_type_L2_original"].astype(object).fillna("Unknown")
)

n_labeled = int((adata.obs[LABELS_KEY] != UNLABELED_CATEGORY).sum())
n_unknown  = int((adata.obs[LABELS_KEY] == UNLABELED_CATEGORY).sum())
print(f"L3 Labeled : {n_labeled:,}  ({n_labeled/adata.n_obs*100:.1f}%)")
print(f"L3 Unknown : {n_unknown:,}  ({n_unknown/adata.n_obs*100:.1f}%)")
print(f"L3 Distinct: {adata.obs[LABELS_KEY].nunique()} classes")
print(adata.obs[LABELS_KEY].value_counts().to_string())
print(f"\nL2 Distinct: {adata.obs['cell_type_L2'].nunique()} classes")
print(adata.obs["cell_type_L2"].value_counts().to_string())

if n_unknown > 0:
    print("\nUnknown by lineage:")
    print(adata.obs.loc[adata.obs[LABELS_KEY]==UNLABELED_CATEGORY, "lineage_source"]
          .value_counts().to_string())

dropped_unknown = 0
if not ALLOW_UNKNOWN and n_unknown > 0:
    if DROP_UNKNOWN_IF_DISALLOWED:
        dropped_unknown = n_unknown
        print(f"\n[INFO] Dropping {dropped_unknown:,} Unknown cells")
        adata = adata[adata.obs[LABELS_KEY] != UNLABELED_CATEGORY].copy()
        adata.obs[LABELS_KEY] = adata.obs[LABELS_KEY].cat.remove_unused_categories()
        n_labeled = adata.n_obs; n_unknown = 0
        print(f"[OK] After drop: {adata.n_obs:,} cells")
    else:
        assert False, (
            f"[X] {n_unknown:,} Unknown cells. "
            f"Set ALLOW_UNKNOWN=True or DROP_UNKNOWN_IF_DISALLOWED=True."
        )
elif ALLOW_UNKNOWN and n_unknown > 0:
    print(f"[INFO] ALLOW_UNKNOWN=True: {n_unknown:,} semi-supervised")

# NOTE: cell_type_final is NOT set here.
# It will be assigned after scANVI in Cell 17.
adata.uns["dropped_unknown_cells"] = int(dropped_unknown)
adata.uns["allow_unknown"]         = bool(ALLOW_UNKNOWN)
print("\n[INFO] cell_type_final will be set after scANVI (Cell 17)")

## Cell 9 — Covariates (MT%)

In [11]:
print("\n" + "="*70)
print("[STEP 9] Covariates")
print("="*70)
mt_mask = adata.var_names.str.startswith("MT-")
n_mt    = int(mt_mask.sum())
_total  = np.array(adata.layers["counts"].sum(axis=1)).ravel().astype(np.float64)
if n_mt > 0:
    _mt = np.array(adata.layers["counts"][:,mt_mask].sum(axis=1)).ravel().astype(np.float64)
    adata.obs["pct_counts_mt"] = (_mt / np.maximum(_total,1) * 100).astype(np.float32)
    del _mt
else:
    adata.obs["pct_counts_mt"] = np.float32(0.0)
    print("[WARNING] No MT- genes found")
del _total
print(f"MT genes: {n_mt} | mean={adata.obs['pct_counts_mt'].mean():.2f}% "
      f"max={adata.obs['pct_counts_mt'].max():.2f}%")

## Cell 10 — Filter Genes, Cells + Drop Small Batches

In [12]:
print("\n" + "="*70)
print("[STEP 10] Filter genes/cells + drop small batches (QRM 16.4 + P2-1)")
print("="*70)
n_c, n_g = adata.n_obs, adata.n_vars
sc.pp.filter_genes(adata, min_cells=3)
sc.pp.filter_cells(adata, min_genes=200)
print(f"Genes : {n_g:,} -> {adata.n_vars:,}")
print(f"Cells : {n_c:,} -> {adata.n_obs:,}")

# [P2-1] Drop batches with <MIN_CELLS_PER_BATCH cells
batch_counts = adata.obs[BATCH_KEY].value_counts()
small_batches = batch_counts[batch_counts < MIN_CELLS_PER_BATCH].index.tolist()
if small_batches:
    n_small_cells = int(adata.obs[BATCH_KEY].isin(small_batches).sum())
    print(f"\n[P2-1] Dropping {len(small_batches)} small batches ({n_small_cells:,} cells): "
          f"{small_batches}")
    adata = adata[~adata.obs[BATCH_KEY].isin(small_batches)].copy()
    print(f"       After drop: {adata.n_obs:,} cells | {adata.obs[BATCH_KEY].nunique()} batches")
else:
    print(f"[OK] All {adata.obs[BATCH_KEY].nunique()} batches pass minimum size ({MIN_CELLS_PER_BATCH})")

## Cell 11 — HVG Selection (4-tier fallback, QRM 16.5)

In [13]:
print("\n" + "="*70)
print("[STEP 11] HVG selection")
print("="*70)
# [QRM 12.5] Clear stale HVG aux cols
_drop = [c for c in _HVG_AUX_VAR_COLS if c in adata.var.columns]
if _drop: adata.var.drop(columns=_drop, inplace=True)

hvg_method = None
try:
    sc.pp.highly_variable_genes(adata, layer="counts", n_top_genes=N_HVG,
                                batch_key=BATCH_KEY, flavor="seurat_v3", subset=False)
    hvg_method = "seurat_v3 batch-aware (counts)"
except Exception as e1:
    print(f"  Tier1 failed: {e1}")
    try:
        sc.pp.highly_variable_genes(adata, layer="counts", n_top_genes=N_HVG,
                                    flavor="seurat_v3", subset=False)
        hvg_method = "seurat_v3 non-batch (counts)"
    except Exception as e2:
        print(f"  Tier2 failed: {e2}")
        # Transient log1p -- deleted after HVG
        _cnt = adata.layers["counts"]
        _tot = np.array(_cnt.sum(axis=1)).ravel().astype(np.float64)
        _sc  = sparse.diags(1e4 / np.maximum(_tot, 1))
        _l1p = (_sc @ _cnt).tocsr(); _l1p.data = np.log1p(_l1p.data).astype(np.float32)
        adata.layers["_log1p_tmp"] = _l1p
        del _cnt, _tot, _sc, _l1p; gc.collect()
        try:
            sc.pp.highly_variable_genes(adata, layer="_log1p_tmp", n_top_genes=N_HVG,
                                        batch_key=BATCH_KEY, flavor="seurat", subset=False)
            hvg_method = "seurat batch-aware (log1p)"
        except Exception as e3:
            print(f"  Tier3 failed: {e3}")
            sc.pp.highly_variable_genes(adata, layer="_log1p_tmp", n_top_genes=N_HVG,
                                        flavor="seurat", subset=False)
            hvg_method = "seurat non-batch (log1p)"
        del adata.layers["_log1p_tmp"]; gc.collect()

n_hvg = int(adata.var["highly_variable"].sum())
assert n_hvg > 0, "[X] Zero HVGs"
adata.uns["hvg_method"] = hvg_method
print(f"[OK] HVG: {n_hvg:,} | {hvg_method}")
pd.Series(adata.var_names[adata.var["highly_variable"]].tolist()).to_csv(
    OUTPUT_DIR/"hvg_genes_final.csv", index=False, header=False)
pd.Series(adata.var_names.tolist()).to_csv(
    OUTPUT_DIR/"all_genes_union.csv", index=False, header=False)
print(f"[OK] gene lists saved (HVG: {n_hvg:,}  union: {adata.n_vars:,})")

## Cell 12 — Save `.raw` (post-concat union gene space, QRM item 2)

In [14]:
print("\n" + "="*70)
print("[STEP 12] Save .raw (union gene space, shared memory, QRM item 2)")
print("="*70)
# [P0-2 fix] With outer join, adata at this point has union gene space (~57k genes).
# This IS a meaningful full-union-gene .raw, not just the inner-join intersection.
# Labeled in run log as 'post_concat_union_raw' to be precise.
adata.raw = sc.AnnData(
    X   = adata.layers["counts"],   # shared memory -- no .copy() [QRM 2]
    obs = adata.obs.copy(),
    var = adata.var.copy(),
)
print(f"[OK] .raw: {adata.raw.n_vars:,} genes (union of all lineages)")
print(f"     Note: covers full outer-join gene space (not each lineage's own full-gene)")

import psutil
print(f"Memory: {psutil.Process(os.getpid()).memory_info().rss/1e9:.1f} GB")

## Cell 13 — Subset to HVG

In [15]:
print("\n" + "="*70)
print("[STEP 13] HVG subset")
print("="*70)
adata = adata[:, adata.var["highly_variable"]].copy()
print(f"[OK] {adata.n_obs:,} x {adata.n_vars:,}")
print(f"Memory: {psutil.Process(os.getpid()).memory_info().rss/1e9:.1f} GB")

## Cell 14 — Normalize `.X` log1p (after HVG subset)

In [16]:
print("\n" + "="*70)
print("[STEP 14] Normalize .X log1p (HVG only)")
print("="*70)
adata.X = adata.layers["counts"].copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.layers["log1p"] = adata.X.copy()
print("[OK] .X = log1p (HVG)")

## Cell 15 — scVI Training

In [17]:
print("\n" + "="*70)
print("[STEP 15] scVI training")
print("="*70)

scvi.model.SCVI.setup_anndata(
    adata, layer="counts", batch_key=BATCH_KEY,
    continuous_covariate_keys=["pct_counts_mt"],
)
model_scvi = scvi.model.SCVI(
    adata, n_latent=N_LATENT, n_layers=2, n_hidden=128, dropout_rate=0.1,
    gene_likelihood="nb", dispersion="gene-batch",
    encode_covariates=True,   # [QRM 15] scArches compatibility
)
# [QRM 7] param count via try-except
try:    n_params = model_scvi.module.n_params
except: n_params = sum(p.numel() for p in model_scvi.module.parameters() if p.requires_grad)
print(f"scVI params: {n_params:,}")

model_scvi.train(
    max_epochs=SCVI_EPOCHS, batch_size=256, train_size=0.9,
    early_stopping=True, plan_kwargs={"lr": 1e-3},
    accelerator=_accelerator, devices=_devices,
)
adata.obsm["X_scvi"] = model_scvi.get_latent_representation()

SCVI_MODEL_PATH = MODEL_DIR / "scvi_model"
model_scvi.save(str(SCVI_MODEL_PATH), overwrite=True)
pd.Series(adata.var_names.tolist()).to_csv(          # [QRM 17]
    SCVI_MODEL_PATH/"var_names.csv", index=False, header=False)

try:                                                   # [QRM 16] dry-run 2k
    _idx = np.random.choice(adata.n_obs, min(2000, adata.n_obs), replace=False)
    _ck  = adata[_idx].copy()
    scvi.model.SCVI.prepare_query_anndata(_ck, str(SCVI_MODEL_PATH))
    del _ck; gc.collect()
    print("[OK] scVI scArches dry-run (2k)")
except Exception as e: print(f"[WARNING] scVI dry-run: {e}")
print(f"[OK] scVI saved: {SCVI_MODEL_PATH}")

## Cell 16 — UMAP (scVI latent)

In [18]:
print("\n" + "="*70)
print("[STEP 16] Neighbors + UMAP (scVI)")
print("="*70)
sc.pp.neighbors(adata, use_rep="X_scvi", n_neighbors=20)
sc.tl.umap(adata, min_dist=0.3)
adata.obsm["X_umap_scvi"] = adata.obsm["X_umap"].copy()
print("[OK] X_umap_scvi saved")

## Cell 17 — scANVI Training + Assign `cell_type_final`

In [19]:
print("\n" + "="*70)
print("[STEP 17] scANVI training (L3 fully supervised)")
print("="*70)

# [P0-5] from_scvi_model inherits registry -- no SCANVI.setup_anndata()
assert LABELS_KEY in adata.obs.columns
if UNLABELED_CATEGORY not in adata.obs[LABELS_KEY].cat.categories:
    adata.obs[LABELS_KEY] = adata.obs[LABELS_KEY].cat.add_categories([UNLABELED_CATEGORY])

model_scanvi = scvi.model.SCANVI.from_scvi_model(
    model_scvi, unlabeled_category=UNLABELED_CATEGORY, labels_key=LABELS_KEY,
)
model_scanvi.train(
    max_epochs=SCANVI_EPOCHS, batch_size=256, train_size=0.9,
    early_stopping=True, plan_kwargs={"lr": 1e-3, "weight_decay": 0.0},
    accelerator=_accelerator, devices=_devices,
)
adata.obsm["X_scanvi"] = model_scanvi.get_latent_representation()

# [QRM 9 + P1-2] label order from predict(soft=True); index-aligned write-back
soft_df = model_scanvi.predict(soft=True)
soft_df = soft_df.reindex(adata.obs_names)   # [P1-2] guarantee row alignment

adata.obs["cell_type_scanvi_pred_pan"] = soft_df.idxmax(axis=1)
adata.obs["scanvi_confidence"]         = soft_df.max(axis=1).astype(np.float32)
adata.obsm["scanvi_probabilities"]     = soft_df.values.astype(np.float32)
adata.uns["scanvi_celltype_order"]     = soft_df.columns.tolist()

# [P1-1] cell_type_final = scANVI output (the refined pan-lineage annotation)
# cell_type_input_l3 retains the original per-lineage label for provenance
adata.obs["cell_type_final"] = adata.obs["cell_type_scanvi_pred_pan"].astype(object)

print(f"[OK] scANVI: {soft_df.shape[1]} L3 classes | "
      f"mean confidence={adata.obs['scanvi_confidence'].mean():.4f}")
print(f"[P1-1] cell_type_final <- cell_type_scanvi_pred_pan (pan-scANVI refined)")
print(f"       cell_type_input_l3 retains original per-lineage labels")

SCANVI_MODEL_PATH = MODEL_DIR / "scanvi_model"
model_scanvi.save(str(SCANVI_MODEL_PATH), overwrite=True)
pd.Series(adata.var_names.tolist()).to_csv(          # [QRM 17]
    SCANVI_MODEL_PATH/"var_names.csv", index=False, header=False)

try:                                                   # [QRM 16] dry-run 2k
    _idx = np.random.choice(adata.n_obs, min(2000, adata.n_obs), replace=False)
    _ck  = adata[_idx].copy()
    scvi.model.SCANVI.prepare_query_anndata(_ck, str(SCANVI_MODEL_PATH))
    del _ck; gc.collect()
    print("[OK] scANVI scArches dry-run (2k)")
except Exception as e: print(f"[WARNING] scANVI dry-run: {e}")

del model_scvi, model_scanvi; gc.collect()           # [QRM 4]
if GPU_AVAILABLE: torch.cuda.empty_cache()
print(f"[OK] scANVI saved: {SCANVI_MODEL_PATH}")

## Cell 18 — UMAP (scANVI) + Agreement Check

In [20]:
print("\n" + "="*70)
print("[STEP 18] Neighbors + UMAP (scANVI) + agreement check")
print("="*70)
sc.pp.neighbors(adata, use_rep="X_scanvi", n_neighbors=20)
sc.tl.umap(adata, min_dist=0.3)
adata.obsm["X_umap_scanvi"] = adata.obsm["X_umap"].copy()

# Agreement: cell_type_final (scANVI pred) vs cell_type_input_l3 (original)
agree = (adata.obs["cell_type_final"].astype(str) ==
         adata.obs["cell_type_input_l3"].astype(str))
agreement_rate = float(agree.mean())
adata.uns["agreement_rate_overall"] = agreement_rate
print(f"Overall L3 agreement (scANVI pred vs input label): {agreement_rate*100:.2f}%")
for lin in adata.obs["lineage_source"].unique():
    ag = agree[adata.obs["lineage_source"]==lin].mean()
    print(f"  {lin:12s}: {ag*100:.2f}%")

## Cell 19 — UMAP Figures (L3 + L2 + lineage + confidence)

In [21]:
print("\n" + "="*70)
print("[STEP 19] UMAP figures")
print("="*70)
# [QRM v3.6 / Sec 11] vector_friendly=True; no rasterized= kwarg
sc.settings.vector_friendly = True

_cols = ["lineage_source", "cell_type_final", "cell_type_L2", BATCH_KEY, "scanvi_confidence"]
for _basis, _lbl in [("X_umap_scvi","scVI"), ("X_umap_scanvi","scANVI")]:
    fig, axes = plt.subplots(1, len(_cols), figsize=(6*len(_cols), 5))
    for ax, col in zip(axes, _cols):
        sc.pl.embedding(adata, basis=_basis, color=col, ax=ax, show=False, title=col,
                        legend_loc="right margin" if adata.obs[col].nunique()<=30 else "none",
                        frameon=False)
    fig.suptitle(f"{_lbl} UMAP — All Lineages (L3 scANVI refined)", y=1.02, fontsize=14)
    fig.tight_layout()
    fig.savefig(FIG_DIR/f"allcells_{_lbl.lower()}_umap_overview.pdf",
                dpi=300, bbox_inches="tight")
    plt.close("all")
    print(f"[OK] {_lbl} UMAP saved")

## Cell 20 — Pre-write Cleanup (QRM v3.5 §17.1-17.3)

In [22]:
print("\n" + "="*70)
print("[STEP 20] Pre-write cleanup")
print("="*70)

# [QRM-34 / v3.5 §17.2] _index reserved column
for _attr in ("obs","var"):
    _df = getattr(adata, _attr)
    if "_index" in _df.columns:
        setattr(adata, _attr, _df.rename(columns={"_index":"orig_index"}))
        print(f"  Renamed {_attr}['_index'] -> 'orig_index'")

# [QRM-35 / v3.5 §17.3] stale obsm
OBSM_KEEP = {"X_scvi","X_scanvi","X_umap_scvi","X_umap_scanvi","scanvi_probabilities"}
stale = [k for k in list(adata.obsm.keys()) if k not in OBSM_KEEP]
for k in stale: del adata.obsm[k]
if stale: print(f"  Removed stale obsm: {stale}")

# [QRM v3.5 §17.1] object-backed category dtype
for col in adata.obs.select_dtypes(include=["category"]).columns:
    cats = adata.obs[col].cat.categories
    if hasattr(cats.dtype,"name") and cats.dtype.name in ("string","StringDtype"):
        adata.obs[col] = adata.obs[col].cat.rename_categories(cats.astype(object))

print("[OK] Pre-write cleanup done")

## Cell 21 — Reference Manifest JSON (QRM item 19)

In [23]:
print("\n" + "="*70)
print("[STEP 21] Reference manifest JSON")
print("="*70)
manifest = {
    "reference_h5ad"      : str(OUTPUT_H5AD),
    "scvi_model_dir"      : str(SCVI_MODEL_PATH),
    "scanvi_model_dir"    : str(SCANVI_MODEL_PATH),
    "hvg_var_names_csv"   : str(SCVI_MODEL_PATH/"var_names.csv"),
    "all_genes_union_csv" : str(OUTPUT_DIR/"all_genes_union.csv"),
    "batch_key"           : BATCH_KEY,
    "batch_granularity"   : "sample",
    "label_key"           : LABELS_KEY,
    "label_tier"          : "L3",
    "l2_key"              : "cell_type_L2",
    "final_label_key"     : "cell_type_final",        # scANVI refined output
    "input_label_key"     : "cell_type_input_l3",     # original per-lineage label
    "prediction_key"      : "cell_type_scanvi_pred_pan",
    "confidence_key"      : "scanvi_confidence",
    "unlabeled_category"  : UNLABELED_CATEGORY,
    "dropped_unknown_cells": int(adata.uns.get("dropped_unknown_cells",0)),
    "gene_space"          : "outer_join_union",
    "n_latent"            : N_LATENT,
    "encode_covariates"   : True,
    "scarches_compatible" : True,
    "agreement_rate"      : float(adata.uns.get("agreement_rate_overall",-1)),
    "version"             : VERSION,
    "date"                : datetime.now().strftime("%Y-%m-%d"),
    "lineage_inputs"      : {
        k: {"h5ad": str(v["h5ad"]), "label_built": "cell_type_L3",
            "batch_local": v["batch_key_local"]}
        for k, v in LINEAGE_INPUTS.items()
    },
}
with open(OUTPUT_DIR/"reference_manifest.json","w") as f:
    json.dump(manifest, f, indent=2)
print("[OK] reference_manifest.json written")

## Cell 22 — Save h5ad

In [24]:
print("\n" + "="*70)
print(f"[STEP 22] Saving {OUTPUT_H5AD.name}")
print("="*70)
adata.write_h5ad(OUTPUT_H5AD, compression="gzip", compression_opts=9)
print(f"[OK] Saved: {OUTPUT_H5AD}")
print(f"     Size : {OUTPUT_H5AD.stat().st_size/1e9:.2f} GB")

TypeError: Can't implicitly convert non-string objects to strings

## Cell 23 — AnnData Structure JSON Output

In [ ]:
print("\n" + "="*70)
print("[STEP 23] AnnData structure JSON")
print("="*70)

n_lab = int((adata.obs[LABELS_KEY].astype(str) != UNLABELED_CATEGORY).sum())
n_unk = int((adata.obs[LABELS_KEY].astype(str) == UNLABELED_CATEGORY).sum())

def _li(X):
    sp = sparse.issparse(X)
    return {"type": type(X).__name__ if sp else "ndarray",
            "shape": list(X.shape), "dtype": str(X.dtype), "sparse": sp}

def _ci(s):
    d = {"dtype": str(s.dtype), "non_null": int(s.notna().sum())}
    try: d["unique"] = int(s.nunique())
    except: pass
    return d

structure = {
    "pipeline"              : PIPELINE_NAME,
    "version"               : VERSION,
    "timestamp"             : datetime.now().isoformat(),
    "output_h5ad"           : str(OUTPUT_H5AD),
    "shape"                 : [adata.n_obs, adata.n_vars],
    "X"                     : _li(adata.X),
    "raw_n_vars"            : int(adata.raw.n_vars) if adata.raw else None,
    "raw_gene_space"        : "post_concat_union (outer join)",   # [P0-2 semantic fix]
    "layers"                : {k: _li(v) for k, v in adata.layers.items()},
    "obsm"                  : {k: {"shape": list(v.shape), "dtype": str(v.dtype)}
                                for k, v in adata.obsm.items()},
    "batch_key"             : BATCH_KEY,
    "batch_granularity"     : "sample",
    "n_batches"             : int(adata.obs[BATCH_KEY].nunique()),
    "gene_space"            : "outer_join_union",
    "label_tier"            : "L3",
    "labels_key"            : LABELS_KEY,
    "n_L3_classes"          : int(adata.obs[LABELS_KEY].astype(str).nunique()),
    "n_L2_classes"          : int(adata.obs["cell_type_L2"].nunique()),
    "n_labeled"             : n_lab,
    "n_unknown"             : n_unk,
    "dropped_unknown"       : int(adata.uns.get("dropped_unknown_cells",0)),
    "hvg_method"            : adata.uns.get("hvg_method","unknown"),
    "n_latent"              : N_LATENT,
    "agreement_rate"        : float(adata.uns.get("agreement_rate_overall",-1)),
    "scanvi_celltype_order" : adata.uns.get("scanvi_celltype_order",[]),
    "key_obs_columns"       : {
        "cell_type_final"           : _ci(adata.obs["cell_type_final"]),
        "cell_type_input_l3"        : _ci(adata.obs["cell_type_input_l3"]),
        "cell_type_L2"              : _ci(adata.obs["cell_type_L2"]),
        LABELS_KEY                  : _ci(adata.obs[LABELS_KEY]),
        "cell_type_scanvi_pred_pan" : _ci(adata.obs["cell_type_scanvi_pred_pan"]),
        "scanvi_confidence"         : _ci(adata.obs["scanvi_confidence"]),
        "lineage_source"            : _ci(adata.obs["lineage_source"]),
        BATCH_KEY                   : _ci(adata.obs[BATCH_KEY]),
    },
    "lineage_distribution"  : adata.obs["lineage_source"].value_counts().to_dict(),
    "L3_final_distribution" : adata.obs["cell_type_final"].value_counts().head(60).to_dict(),
    "L2_distribution"       : adata.obs["cell_type_L2"].value_counts().to_dict(),
    "confidence_stats"      : {k: float(v)
                                for k,v in adata.obs["scanvi_confidence"].describe().items()},
    "log_file"              : str(LOG_FILE),
}
struct_path = OUTPUT_DIR / f"anndata_structure_{DATE_TAG}_v{VERSION}.json"
with open(struct_path,"w") as f: json.dump(structure, f, indent=2)
print(f"[OK] Structure JSON: {struct_path}")
print(f"  shape           : {adata.n_obs:,} x {adata.n_vars:,}")
print(f"  .raw            : {adata.raw.n_vars:,} genes (union space)")
print(f"  layers          : {list(adata.layers.keys())}")
print(f"  obsm            : {list(adata.obsm.keys())}")
print(f"  L3 classes      : {structure['n_L3_classes']}  |  L2: {structure['n_L2_classes']}")
print(f"  agreement       : {float(adata.uns.get('agreement_rate_overall',-1))*100:.2f}%")

## Cell 24 — Pipeline Run Log + Close Log File

In [ ]:
elapsed_min = (time.time() - PIPELINE_START) / 60
n_lab = int((adata.obs[LABELS_KEY].astype(str) != UNLABELED_CATEGORY).sum())
n_unk = int((adata.obs[LABELS_KEY].astype(str) == UNLABELED_CATEGORY).sum())

_log = [
    "="*80, f"{PIPELINE_NAME} Run Log  (v{VERSION})", "="*80,
    f"written_at              : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    f"log_file                : {LOG_FILE}",
    f"output_h5ad             : {OUTPUT_H5AD}",
    f"n_cells                 : {adata.n_obs:,}",
    f"n_genes_hvg             : {adata.n_vars:,}",
    f"n_genes_post_concat_raw : {adata.raw.n_vars if adata.raw else 'None'}",
    f"  (= union outer join gene space; not per-lineage full-gene)",
    f"n_batches               : {adata.obs[BATCH_KEY].nunique():,}",
    f"batch_granularity       : sample",
    f"concat_join             : outer (P0-1 fix; gene space = union ~57k)",
    f"hvg_method              : {adata.uns.get('hvg_method','unknown')}",
    f"n_latent                : {N_LATENT}",
    f"label_tier              : L3",
    f"cell_type_final         : scANVI output (pan-refined)  [P1-1 fix]",
    f"cell_type_input_l3      : original per-lineage label",
    f"n_labeled               : {n_lab:,}",
    f"n_unknown               : {n_unk:,}",
    f"dropped_unknown         : {int(adata.uns.get('dropped_unknown_cells',0)):,}",
    f"agreement_rate          : {_fmt_float(adata.uns.get('agreement_rate_overall'))}",
    f"accelerator             : {_accelerator}",
    f"elapsed_minutes         : {elapsed_min:.2f}",
    "",
    "lineage_source distribution:",
    adata.obs["lineage_source"].value_counts().to_string(),
    "",
    "cell_type_L2 distribution:",
    adata.obs["cell_type_L2"].value_counts().to_string(),
    "",
    "cell_type_final (scANVI refined L3) distribution (top 60):",
    adata.obs["cell_type_final"].value_counts().head(60).to_string(),
    "",
    "cell_type_input_l3 (original labels) distribution (top 40):",
    adata.obs["cell_type_input_l3"].value_counts().head(40).to_string(),
    "",
    "scanvi_confidence percentiles:",
    adata.obs["scanvi_confidence"].describe().to_string(),
    "",
    "label build source per lineage:",
]
_label_src = {
    "epithelial": "scanvi_fine_pred + Hillock-like/Ionocyte/Neuroendocrine override",
    "tcell"     : "scanvi_label_refined (CD4 Naive/TCM -> CD4 Naive)",
    "myeloid"   : "ann_finest_level + cell_type_L3_refined fallback; hard-fail on unmapped",
    "bcell"     : "cell_type_scanvi_pred (IGHEplus -> Atypical_Memory_B)",
    "stromal"   : "cell_type_scanvi_pred",
}
for k, v in _label_src.items():
    _log.append(f"  {k:12s}: {v}")

(OUTPUT_DIR/"pipeline_run_log.txt").write_text("\n".join(_log)+"\n", encoding="utf-8")
(OUTPUT_DIR/"anndata_structure.txt").write_text(
    f"See anndata_structure_{DATE_TAG}_v{VERSION}.json\n"
    f"n_obs={adata.n_obs:,} n_vars={adata.n_vars:,}\n", encoding="utf-8")

print("="*70)
print("[DONE] Pipeline complete")
print(f"Elapsed  : {elapsed_min:.2f} min")
print(f"Output   : {OUTPUT_H5AD}")
print(f"Log      : {LOG_FILE}")
print("="*70)

# Close Tee
try:
    sys.stdout = sys.__stdout__
    sys.stderr = sys.__stderr__
except Exception: pass
try:
    if "_log_fh" in globals() and not getattr(_log_fh,"closed",True):
        _log_fh.flush(); _log_fh.close()
        print(f"[OK] Log closed: {LOG_FILE}")
except Exception as e:
    print(f"[WARNING] Log close: {e}")